In [0]:
%run ./logger


### Step 1: Import Libraries

In [0]:
from datetime import datetime
import time
import uuid

In [0]:
batch_id = dbutils.widgets.get("batch_id")


In [0]:

start_time = log_start(
    batch_id,
    "raw_data",
    "raw",
    "bronze"
)

### Step 2: Create Database

### Step 3: Create Execution Log Table

database,tableName,isTemporary
medallion_db,execution_logs,false


### Step 4: Start Logger Function

### Step 5: Success Logger Function

In [0]:
spark.conf.set(
    "fs.azure.account.key.insurancedatahomecredit.dfs.core.windows.net",
    "mvIlwSRtmT+AXedlxE40hoBJZsEQlvxnlHDD1ZdWje1R7m00TeP/QXcK1Gr7Nu8os0Mru1pMnb+E+AStH5S9mA==")

### Step 6: Failure Logger Function

### Step 7: View Execution Logs

Batch_ID,Notebook_Name,Source_Layer,Target_Layer,Start_Time,End_Time,Duration_Seconds,Status,Records_Read,Records_Written,Error_Message
,gold_data,silver,gold,2026-06-27T12:50:19.283856Z,2026-06-27T12:51:11.528412Z,52,SUCCESS,1017209,1017209,
4f5195ce-99d6-407a-9bf5-2991af90f714,raw_data,raw,bronze,2026-06-27T12:49:52.171746Z,2026-06-27T12:50:19.464848Z,27,SUCCESS,1018324,1018324,


### 

In [0]:
spark.conf.set(
    "fs.azure.account.key.insurancedatahomecredit.blob.core.windows.net",
    spark.conf.get("fs.azure.account.key.insurancedatahomecredit.dfs.core.windows.net")
)

train_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("wasbs://raw@insurancedatahomecredit.blob.core.windows.net/train.csv")

display(train_df)

Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
1,5,2015-07-31,5263,555,1,1,0,1
2,5,2015-07-31,6064,625,1,1,0,1
3,5,2015-07-31,8314,821,1,1,0,1
4,5,2015-07-31,13995,1498,1,1,0,1
5,5,2015-07-31,4822,559,1,1,0,1
6,5,2015-07-31,5651,589,1,1,0,1
7,5,2015-07-31,15344,1414,1,1,0,1
8,5,2015-07-31,8492,833,1,1,0,1
9,5,2015-07-31,8565,687,1,1,0,1
10,5,2015-07-31,7185,681,1,1,0,1


In [0]:
store_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("wasbs://raw@insurancedatahomecredit.blob.core.windows.net/store.csv")

display(store_df)

Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
1,c,a,1270,9,2008,0,null,null,null
2,a,a,570,11,2007,1,13,2010,"Jan,Apr,Jul,Oct"
3,a,a,14130,12,2006,1,14,2011,"Jan,Apr,Jul,Oct"
4,c,c,620,9,2009,0,null,null,null
5,a,a,29910,4,2015,0,null,null,null
6,a,a,310,12,2013,0,null,null,null
7,a,c,24000,4,2013,0,null,null,null
8,a,a,7520,10,2014,0,null,null,null
9,a,c,2030,8,2000,0,null,null,null
10,a,a,3160,9,2009,0,null,null,null


In [0]:
train_df.printSchema()
store_df.printSchema()


root
 |-- Store: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Sales: integer (nullable = true)
 |-- Customers: integer (nullable = true)
 |-- Open: integer (nullable = true)
 |-- Promo: integer (nullable = true)
 |-- StateHoliday: string (nullable = true)
 |-- SchoolHoliday: integer (nullable = true)

root
 |-- Store: integer (nullable = true)
 |-- StoreType: string (nullable = true)
 |-- Assortment: string (nullable = true)
 |-- CompetitionDistance: integer (nullable = true)
 |-- CompetitionOpenSinceMonth: integer (nullable = true)
 |-- CompetitionOpenSinceYear: integer (nullable = true)
 |-- Promo2: integer (nullable = true)
 |-- Promo2SinceWeek: integer (nullable = true)
 |-- Promo2SinceYear: integer (nullable = true)
 |-- PromoInterval: string (nullable = true)



In [0]:
train_count = train_df.count()
store_count = store_df.count()

#### batch_id

#### start time

#### Capture Start Time

Add Metadata Columns to Train File

In [0]:
from pyspark.sql.functions import *

train_df = train_df \
.withColumn("batch_id", lit(batch_id)) \
.withColumn("ingestion_timestamp", current_timestamp()) \
.withColumn("source_file", lit("train.csv"))

Add Metadata Columns to Store File

In [0]:
from pyspark.sql.functions import *

store_df = store_df\
    .withColumn("batch_id", lit(batch_id))\
    .withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("source_file", lit("store_df"))

Empty File Validation

In [0]:
if train_count == 0:
    raise Exception("Train file is empty")

if store_count == 0:
    raise Exception("Store file is empty")

In [0]:
train_duplicates = train_count - train_df.dropDuplicates().count()

store_duplicates = store_count - store_df.dropDuplicates().count()

print("Train Duplicates :", train_duplicates)

print("Store Duplicates :", store_duplicates)

Train Duplicates : 0
Store Duplicates : 0


In [0]:
train_df.write \
.format("delta") \
.mode("overwrite") \
.save("wasbs://bronze@insurancedatahomecredit.blob.core.windows.net/train")

In [0]:
store_df.write \
.format("delta") \
.mode("overwrite") \
.save("wasbs://bronze@insurancedatahomecredit.blob.core.windows.net/store")

In [0]:
train_count = train_df.count()
store_count = store_df.count()

records_read = train_count + store_count

In [0]:
records_written = train_count + store_count

In [0]:
log_success(
    batch_id,
    "raw_data",
    "raw",
    "bronze",
    start_time,
    records_read,
    records_written
)